In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor()
]))
test_data = datasets.MNIST(root='./data', train=False, download=True, transform=transforms.ToTensor())
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

print(len(train_data), train_data[0][0].shape)

60000 torch.Size([1, 28, 28])


: 

In [ ]:
class SimpleNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.relu = nn.ReLU()
        # self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, 10)
    
    def forward(self, x):
        x = x.view(x.size(0), -1)  # Flatten (B, 1, 28, 28) → (B, 784)
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu(x)
        # x = self.dropout(x)
        x = self.fc2(x)
        return x

model = SimpleNN()
print(model)

# Loss
criterion = nn.CrossEntropyLoss()
optimizer = optim.RMSprop(model.parameters(), lr=1e-3, alpha=0.9, weight_decay=1e-4)

def train(model, train_loader, epochs=5):
    # Enables training mode for the model
    model.train()
    
    for epoch in range(epochs):
        total_loss = 0
        correct = 0
        # Minibatch training
        for images, labels in train_loader:
            # print(images.shape, labels.shape)
            # 0 the gradients
            optimizer.zero_grad()
            # Forward pass
            outputs = model(images)
            # compute loss
            loss = criterion(outputs, labels)
            # Backward pass and compute gradients
            loss.backward()
            # update weights
            # optimizer.step() is called to update the model parameters based on the computed gradients
            optimizer.step()
            
            total_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader)}, Accuracy: {correct/len(train_data)}")
        
def train_and_evaluate(model, train_loader, test_loader, train_data, test_data, epochs=5):
    model.train()
    
    train_losses, test_losses = [], []
    train_accuracies, test_accuracies = [], []

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        correct = 0

        for images, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
        
        avg_train_loss = total_loss / len(train_loader)
        train_accuracy = correct / len(train_data)

        # Evaluate on test set
        model.eval()
        with torch.no_grad():
            total_test_loss = 0
            correct = 0
            for images, labels in test_loader:
                outputs = model(images)
                loss = criterion(outputs, labels)
                total_test_loss += loss.item()
                preds = outputs.argmax(dim=1)
                correct += (preds == labels).sum().item()
        
        avg_test_loss = total_test_loss / len(test_loader)
        test_accuracy = correct / len(test_data)

        # Log
        train_losses.append(avg_train_loss)
        test_losses.append(avg_test_loss)
        train_accuracies.append(train_accuracy)
        test_accuracies.append(test_accuracy)

        print(f"Epoch {epoch+1}: Train Loss={avg_train_loss:.4f}, Test Loss={avg_test_loss:.4f}, Train Acc={train_accuracy:.4f}, Test Acc={test_accuracy:.4f}")

    print(f"maximum test accuracy: {max(test_accuracies):.4f} at epoch {test_accuracies.index(max(test_accuracies)) + 1}")
    print(f"maximum train accuracy: {max(train_accuracies):.4f} at epoch {train_accuracies.index(max(train_accuracies)) + 1}")
    
    # Plot
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='Train Loss')
    plt.plot(test_losses, label='Test Loss')
    plt.title('Loss Curve')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(train_accuracies, label='Train Accuracy')
    plt.plot(test_accuracies, label='Test Accuracy')
    plt.title('Accuracy Curve')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.tight_layout()
    plt.show()

train_and_evaluate(model, train_loader, test_loader, train_data, test_data, epochs=20)

# train(model, train_loader, epochs=5)

SimpleNN(
  (fc1): Linear(in_features=784, out_features=256, bias=True)
  (bn1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU()
  (fc2): Linear(in_features=256, out_features=10, bias=True)
)
Epoch 1: Train Loss=0.4949, Test Loss=0.1708, Train Acc=0.8561, Test Acc=0.9536
Epoch 2: Train Loss=0.2615, Test Loss=0.1153, Train Acc=0.9253, Test Acc=0.9685
Epoch 3: Train Loss=0.2156, Test Loss=0.1070, Train Acc=0.9366, Test Acc=0.9694
Epoch 4: Train Loss=0.1962, Test Loss=0.0900, Train Acc=0.9423, Test Acc=0.9756
Epoch 5: Train Loss=0.1846, Test Loss=0.0829, Train Acc=0.9439, Test Acc=0.9737
Epoch 6: Train Loss=0.1742, Test Loss=0.0770, Train Acc=0.9487, Test Acc=0.9776
Epoch 7: Train Loss=0.1677, Test Loss=0.0811, Train Acc=0.9493, Test Acc=0.9755
Epoch 8: Train Loss=0.1666, Test Loss=0.0743, Train Acc=0.9507, Test Acc=0.9776
Epoch 9: Train Loss=0.1613, Test Loss=0.0706, Train Acc=0.9514, Test Acc=0.9782
Epoch 10: Train Loss=0.1555, Test Loss

In [ ]:
def visualize_batchnorm_effect(model, data_loader):
    model.eval()
    with torch.no_grad():
        for images, labels in data_loader:
            x = images.view(images.size(0), -1)
            x_fc = model.fc1(x)               # Before BatchNorm
            x_relu = F.relu(x_fc)             # After ReLU
            x_bn = model.bn1(x_fc)          # After BatchNorm
            
            # Plot histograms
            plt.figure(figsize=(12, 5))
            
            plt.subplot(1, 2, 1)
            plt.hist(x_relu.flatten().numpy(), bins=100, color='skyblue')
            plt.title('Before BatchNorm (ReLU Output)')
            plt.xlabel('Activation Value')
            plt.ylabel('Frequency')

            plt.subplot(1, 2, 2)
            plt.hist(x_bn.flatten().numpy(), bins=100, color='lightgreen')
            plt.title('After BatchNorm')
            plt.xlabel('Activation Value')
            plt.ylabel('Frequency')

            plt.tight_layout()
            plt.show()
            break  # Visualize just one batch

visualize_batchnorm_effect(model, test_loader)

NameError: name 'F' is not defined

In [ ]:
def predict(model, image_tensor):
    model.eval()
    with torch.no_grad():
        image_tensor = image_tensor.view(1, -1)
        output = model(image_tensor)
        pred = output.argmax(dim=1).item()
    return pred

In [ ]:
sample_img, sample_label = test_data[0]
print(f"predicted: {predict(model, sample_img)}, actual: {sample_label}")

predicted: 7, actual: 7


In [ ]:
def evaluate(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    accuracy = correct / total
    print(f"Test Accuracy: {accuracy:.4f}")
    return accuracy


In [ ]:
evaluate(model, test_loader)

Test Accuracy: 0.9789


0.9789